# DaliScope quickstart

Load a frozen DALI result pack, explore hits, define a population, and save your analysis.

In **Google Colab**, run the setup cell below to install DaliScope automatically. In **local Jupyter**, install using the root README and select **Python (DaliScope)**. Run these cells from top to bottom. All example outputs are written to `results/quickstart/`.

The included example uses query **3ubpC** (570 residues) and 1,048 packed targets. Raw search alignments and retained view rows can have different counts.

In [ ]:
# In Colab this installs DaliScope automatically. Locally, follow the README first.
import sys
from pathlib import Path
import subprocess
import runpy

try:
    import google.colab
except ModuleNotFoundError as error:
    if error.name not in {"google", "google.colab"}:
        raise
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    DALISCOPE_RELEASE = "v0.1.6"
    colab_root = Path("/content/DaliScope")
    if not colab_root.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", DALISCOPE_RELEASE,
            "https://github.com/Liisa-Holm-group/DaliScope.git", str(colab_root),
        ], check=True)
    setup_path = colab_root / "scripts/colab_setup.py"
    if not setup_path.is_file():
        raise RuntimeError("This runtime contains an older checkout. Start a fresh runtime and run setup again.")
    setup = runpy.run_path(str(setup_path))
    setup["setup_colab"](colab_root, DALISCOPE_RELEASE)
else:
    print("Local Jupyter: using the installed DaliScope environment.")


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / "daliscope").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "daliscope").is_dir():
    raise RuntimeError("Start this notebook from the DaliScope root or notebooks directory")
os.chdir(ROOT)

import daliscope
from daliscope.core.project import Project
%matplotlib inline
import matplotlib.pyplot as plt

print("DaliScope", daliscope.__version__)
pack_path = ROOT / "notebooks/data/3ubpC_PDB25.tar.gz"
output = ROOT / "results/quickstart"
output.mkdir(parents=True, exist_ok=True)

## 1. Load the example

To analyze your own pack, change `pack_path` above. The archive must follow the documented [data format](../docs/data-format.md).

In [ ]:
project = Project.load_pack(str(pack_path))
FULL_VIEW = next(iter(project.views))
FULL_DF = project.views[FULL_VIEW]
print("Query:", project.query_identifier)
print("Query length:", project.query_length)
print("Default view:", FULL_VIEW, "alignments:", len(FULL_DF))
display(FULL_DF[["target_id", "z_score", "query_coverage", "rmsd", "pfam"]].head())

## 2. Explore the search landscape

Z-score is the original DALI score. Coverage is the aligned fraction of the full query. The plot does not establish a biological family boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
points = ax.scatter(FULL_DF["z_score"], FULL_DF["query_coverage"],
                    c=FULL_DF["sequence_identity"], cmap="viridis", s=12, alpha=0.65)
ax.set(xlabel="DALI Z-score", ylabel="Query coverage", title="DaliScope: query 3ubpC")
fig.colorbar(points, ax=ax, label="Aligned sequence identity")
fig.tight_layout()
fig.savefig(output / "hit-landscape.png", dpi=160)
plt.show()

## 3. Register a filtered population

This score cutoff is an illustrative user choice. Inspect structures and annotations to choose a suitable cutoff for your own query.

In [ ]:
project.add_subset("z8", FULL_DF["z_score"] >= 8, FULL_VIEW,
                   function="z_score_filter", parameters={"z_score_min": 8})
print("Selected alignments:", len(project.views["z8"]))
project.viz.architecture("z8", renderer="matplotlib", max_targets=8)

## 4. Try a clipped domain

The clipping API uses zero-based, half-open ranges. `0-150` selects displayed query residues 1 through 150. This is a demonstration interval; choose scientifically appropriate boundaries for your own protein.

Domain clipping recomputes view metrics using a uniform least-squares fit over the retained aligned residues. The fold subset then filters these rows using an occupancy-weighted coverage score; it inherits their fitted coordinates and metrics. The original DALI Z-score is retained throughout.

In [ ]:
from daliscope.mechanics.metrics import register_multiple_domains
from daliscope.analysis.occupancy import create_fold_subset

register_multiple_domains("0-150", project, custom_name="example_domain")
create_fold_subset(project, "example_domain", cutoff=0.5)
display(project.list_inventory())

## 5. Inspect a structure in the browser

The query and target traces can be rotated and zoomed. This cell emits browser widgets; interaction requires a notebook frontend and, for some viewers, JavaScript CDN access. Headless execution checks Python rendering, not mouse interaction.

In [ ]:
from daliscope.widgets.pairwise_py3Dmol import generate_superimposition_view

generate_superimposition_view(
    target_ix=0, df=project.views["z8"], project=project,
    q_ca_coords=project.query_ca_coords, target_ca_coords=project.coords,
)

## 6. Export and resume

This example replaces its own previous `results/quickstart/session` output on rerun. The saved session keeps a reference to the original pack, so retain that file at the recorded path. Only open session files you trust.

In [ ]:
FULL_DF[["target_id", "z_score", "query_coverage", "rmsd", "pfam"]].to_csv(
    output / "hits.tsv", sep="\t", index=False,
)
project.save(str(output / "session"), overwrite=True)
restored = Project.open(str(output / "session"))
assert restored.views.keys() == project.views.keys()
print("Saved outputs:", output)
print("Restored views:", list(restored.views))

## Continue your analysis

- [Usage guide](../docs/usage.md): parameters, exports, and troubleshooting.
- `RunDaliScope.ipynb`: comprehensive interactive workflow.
- `WorkedExample1.ipynb`: multidomain and motif case study.
- `WorkedExample2.ipynb`: advanced GOLD structure communities (requires the optional GOLD pack).

Contact: hao.liu@helsinki.fi.